# Fairness Analysis — Group-Conditional FPR & Error Analysis

**Context:** A model trained on English hate speech (Davidson 2017) applied zero-shot to Roman Urdu.
We analyse whether the model's false positive rate varies across demographic groups mentioned in the text.

Groups analysed:
- **Gender:** posts mentioning women (Roman Urdu + English terms)
- **Religion:** posts mentioning religious communities (common in Urdu hate speech)
- **Ethnicity:** posts mentioning ethnic/regional groups

In [ ]:
import sys, os, json
os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from src.evaluator import fairness_analysis, GROUP_LEXICONS

sns.set_theme(style='whitegrid')
os.makedirs('outputs/figures', exist_ok=True)

In [ ]:
# ── Load XLM-R Large zero-shot Urdu predictions ──────────────────────────
pred_path = 'outputs/predictions/xlmr_large_zero_shot_ur_predictions.json'

if not os.path.exists(pred_path):
    raise FileNotFoundError(f'{pred_path} not found — run scripts/zero_shot_eval.py first')

with open(pred_path) as f:
    records = json.load(f)

df = pd.DataFrame(records)
df = df[df['true_label'] != -1]
print(f'Loaded {len(df):,} predictions')
print(f'True hate: {df["true_label"].sum():,}  Predicted hate: {df["predicted_label"].sum():,}')

In [ ]:
# ── Group-conditional FPR ─────────────────────────────────────────────────
results = fairness_analysis(pred_path)

print('\nGroup-Conditional Fairness Metrics:')
print(f'  {"Group":<15} {"Count":>8} {"FPR":>8} {"Hate F1":>10}')
print('-' * 47)
for group, m in results.items():
    fpr = f"{m['fpr']:.4f}"      if m['fpr']     is not None else 'N/A'
    f1  = f"{m['f1_hate']:.4f}"  if m['f1_hate'] is not None else 'N/A'
    print(f'  {group:<15} {m["count"]:>8} {fpr:>8} {f1:>10}')

In [ ]:
# ── Figure: FPR by group ──────────────────────────────────────────────────
overall_fpr = results.get('overall', {}).get('fpr', 0)
groups = [g for g in results if g != 'overall' and results[g]['fpr'] is not None and results[g]['count'] > 0]

if groups:
    fpr_vals = [results[g]['fpr'] for g in groups]
    colors   = ['#e74c3c' if v > overall_fpr * 1.1 else '#2ecc71' for v in fpr_vals]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(groups, fpr_vals, color=colors, alpha=0.85)
    ax.axvline(overall_fpr, color='black', linestyle='--', linewidth=1.5,
               label=f'Overall FPR = {overall_fpr:.3f}')
    ax.set_xlabel('False Positive Rate')
    ax.set_title('Group-Conditional FPR — XLM-R Large Zero-Shot (EN→Urdu)\n'
                 'Red bars = significantly higher than overall FPR')
    ax.legend()
    plt.tight_layout()
    plt.savefig('outputs/figures/group_fpr.png', dpi=150)
    plt.show()

In [ ]:
# ── Error breakdown ───────────────────────────────────────────────────────
fp_df = df[(df['true_label']==0) & (df['predicted_label']==1)]
fn_df = df[(df['true_label']==1) & (df['predicted_label']==0)]
tp_df = df[(df['true_label']==1) & (df['predicted_label']==1)]
tn_df = df[(df['true_label']==0) & (df['predicted_label']==0)]

print(f'TP={len(tp_df):,}  TN={len(tn_df):,}  FP={len(fp_df):,}  FN={len(fn_df):,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, sub, lbl, col in [
    (axes[0], fp_df, 'False Positives\n(non-hate → hate)', '#e74c3c'),
    (axes[1], fn_df, 'False Negatives\n(hate → non-hate)', '#e67e22'),
]:
    if len(sub) == 0: ax.set_title(f'{lbl}\n(none)'); continue
    ax.hist(sub['confidence_hate'], bins=20, color=col, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Model confidence (hate class)')
    ax.set_ylabel('Count')
    ax.set_title(lbl)
    ax.set_xlim(0, 1)
plt.suptitle('Confidence Distribution on Errors — XLM-R Large Zero-Shot (EN→Urdu)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/confidence_error_analysis.png', dpi=150)
plt.show()

In [ ]:
# ── Sample false positives ────────────────────────────────────────────────
print('=== Top False Positives (non-hate predicted as hate, highest confidence) ===\n')
for _, row in fp_df.sort_values('confidence_hate', ascending=False).head(15).iterrows():
    print(f'  [conf={row["confidence_hate"]:.3f}] {str(row["text"])[:120]}')
    print()

In [ ]:
# ── Sample false negatives ────────────────────────────────────────────────
print('=== Top False Negatives (hate predicted as non-hate, lowest confidence) ===\n')
for _, row in fn_df.sort_values('confidence_hate').head(15).iterrows():
    print(f'  [conf={row["confidence_hate"]:.3f}] {str(row["text"])[:120]}')
    print()

## Discussion Points for Report

**Why FPR may be elevated for religious/ethnic groups in Urdu:**
- The English training data (Davidson 2017) is dominated by US-context hate speech (racial slurs, misogyny)
- Cross-lingual transfer carries over EN hate speech indicators but **not** Urdu cultural context
- Religious group mentions in Urdu are very common in neutral text; the model may incorrectly flag them

**Common false positive patterns in Roman Urdu:**
- Counter-speech ('ye galat hai' = 'this is wrong') mentioning slurs
- Reporting/quoting hateful speech in news-style
- Casual use of terms the EN model learned as hate markers
- Code-switching (Urdu words mid-English sentence) confusing the model

**Common false negative patterns:**
- Implicit/coded Urdu hate speech with no English-recognisable signals
- Cultural metaphors and proverbs used as insults
- Short posts lacking context

**Labelling bias discussion:**
- Davidson et al. (2017) annotations reflect US social context
- Roman Urdu annotations from Pakistani annotators reflect a different social context
- The model transfers syntax/vocabulary patterns but **not** social norms — this is the fundamental challenge of cross-lingual hate speech detection